# Example 6: Non-symmetric evolutionary variational inequality

This example uses a projected fixed-point residual for the lower-level problem. The paper reports the state and control at the end of bilevel training.

In [ ]:
from pathlib import Path
import sys

PACKAGE_ROOT = Path.cwd().resolve()
if not (PACKAGE_ROOT / "utils.py").is_file():
    raise FileNotFoundError("Run this notebook from the final_code package directory.")
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import matplotlib.pyplot as plt
import torch
from   torch.autograd import grad
from utils import *
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,TensorDataset
from scipy.stats.qmc import LatinHypercube
from functools import partial
from torch.optim.lr_scheduler import LambdaLR
import os
import time 
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import clear_output
from datetime import datetime
ROOT = PACKAGE_ROOT / "outputs" / "example6"

## Reload nety

In [ ]:
class net_y_res(nn.Module):
    def __init__(self, source, hidden=64, layers=3, init=True, activation='Tanh', hard_=None, obstacle=None):
        super(net_y_res, self).__init__()
        
        # Choose activation function based on activation_flag
        self.activation = activation
        if activation == 'Tanh':
            self.activation = nn.Tanh()
        elif activation == 'Identity':
            self.activation = nn.Identity()
        elif activation == 'Square':
            self.activation = SquaredReLU()
        elif activation == 'Cubic':
            self.activation = CubicReLU()
        elif activation == 'SELU':
            self.activation = nn.SELU()
        elif activation == 'Swish':
            self.activation = nn.SiLU()
        elif activation == 'ReLU':
            self.activation = nn.ReLU()
        
        
        # Define padding size and apply zero-padding to inputs
        self.padding_size = hidden - 2  # Padding size is hidden - 2 based on your code
        
        # Residual blocks
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(hidden, self.activation) for _ in range(layers - 1)]
        )
        
        # Output layer
        self.output_layer = nn.Linear(hidden, 1)
        
        self.hard = hard_
            
        self.obstacle = obstacle  
        if init:
            self._initialize_weights()

    def _initialize_weights(self):
        # Initialize weights for the output layer
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        
    def forward(self, x):        
        # Apply zero-padding to the input tensor
        padded_x = torch.cat([x, torch.zeros(x.size(0), self.padding_size).to(x.device)], dim=1)
        
        # Apply hard constraint (if provided)
        x1 = padded_x[:, 0:1]
        x2 = padded_x[:, 1:2]
        
        if self.hard is None:
            hard = x1 * (1 - x1) * x2 * (1 - x2)
        else:
            hard = self.hard(x1, x2)
        
        # Directly pass through residual blocks
        out = padded_x
        for res_block in self.res_blocks:
            out = res_block(out)
        
        # Output layer
        out = self.output_layer(out)
        out = out * hard
  
        return out

In [ ]:
class Source7:
    """
    Example 6: nonsymmetric operator A = -Δ + b·∇
    b = (1, -1), f(x1,x2) = 10 sin(2π x2 + x1),
    y_d(x1,x2) = x1(1-x1)x2(1-x2), nu = 0.01.
    """
    def __init__(self, device='cpu'):
        self.device = device
        self.b = torch.tensor((1, -1), dtype=torch.float32, device=device).view(2)
        self.nu = 0.01
        self.analystic = False  # no exact/true solution
        self.f = self.compute_f
        self.yd = self.compute_yd

    def compute_f(self, x):
        x1 = x[:, 0:1]
        x2 = x[:, 1:2]
        return 10.0 * (torch.sin(2.0 * torch.pi * x2 + x1))

    def compute_yd(self, x):
        x1 = x[:, 0:1]
        x2 = x[:, 1:2]
        term1 = x1 * (1.0 - x1) * x2 * (1.0 - x2)
        return term1


# Upper-level objective and lower-level residual

In [ ]:
def J_u(source,net_y,net_v,x):
    """
    Objective function:
    J(y, v) = 1/2 * ||y - z_d||^2 + (V/2) * ||v||^2
    Use MSELoss to approximate the integrals.
    """
    yd = source.yd
    nu = source.nu
    term1 = (1/2)*torch.mean((net_y(x)-yd(x))**2)
    term2 = (nu/2) * torch.mean(net_v(x)**2)

    return term1 +term2



def J_l(source, net_y, net_v, x, eta):
    f = source.f
    b = source.b  # torch tensor of shape (2,)
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)

    f_val = f(x)
    y_pred  = net_y(x)
    v_pred  = net_v(x)

    # First derivatives
    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]

    # Second derivatives for Laplacian
    d2y_dx1 = grad(y_x1.sum(), x1, create_graph=True)[0]
    d2y_dx2 = grad(y_x2.sum(), x2, create_graph=True)[0]

    # A y = -Δy + b·∇y
    Ay = -(d2y_dx1 + d2y_dx2) + b[0] * y_x1 + b[1] * y_x2

    # Prox iteration residual for projection onto K = {w >= 0}
    argument = y_pred - eta * Ay + eta * (f_val + v_pred)
    prox_result = torch.relu(argument)

    return torch.mean((prox_result - y_pred).pow(2))



# Relative errors

In [ ]:
def get_xi(source, net_y, net_v, x):
    """
    State: -Δy + b·∇y - v - xi = f  =>  xi = -Δy + b·∇y - v - f
    """
    b = source.b
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)
    y_pred  = net_y(x)
    v_pred  = net_v(x)
    f_val = source.f(x)

    # First derivatives
    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]
    # Second derivatives
    d2y_dx1 = grad(y_x1.sum(), x1, create_graph=True)[0]
    d2y_dx2 = grad(y_x2.sum(), x2, create_graph=True)[0]
    neg_lap_y = -(d2y_dx1 + d2y_dx2)

    xi = neg_lap_y + b[0] * y_x1 + b[1] * y_x2 - v_pred - f_val
    return xi

def ortho_loss(source,net_y, net_v, x):
    """
    Require y and xi to be orthogonal in a chosen metric.
    """
    y_pred  = net_y(x)
    xi = get_xi(source, net_y, net_v, x)

    return torch.mean((y_pred*xi)**2)

def non_neg_loss(source,net_y,net_v, x):
    """
    Require xi to be nonnegative.
    """
    xi = get_xi(source, net_y, net_v, x)
    return torch.mean((torch.relu(-xi))**2)

# Random sampling in the domain

In [ ]:
class RandomDataLoader:
    """
    Random data loader with dual_sampling for independent samples of y/theta and v.
    """
    def __init__(self, batch_size=64, steps_per_epoch=4, dims=2, device='cpu',
                 dual_sampling=False):
        """
        Initialize the random data loader.
        
        Parameters:
            batch_size (int): Target total batch size.
            steps_per_epoch (int): Number of batches per epoch.
            dims (int): Input dimension (default: 2, for x1 and x2).
            device (str): Device for data generation (e.g. 'cpu' or 'cuda:0').
            dual_sampling (bool): Whether to return independent samples for y/theta and v.
        """
        self.steps_per_epoch = steps_per_epoch
        self.dims = dims
        self.device = device
        self.current_step = 0
        

        self.interior_points = max(1, batch_size)
        

        self.batch_size = self.interior_points
        self.dual_sampling = dual_sampling
        
    def __iter__(self):
        """Return the iterator itself."""
        self.current_step = 0
        return self
    
    def __next__(self):
        """Generate a full batch on each call."""
        if self.current_step < self.steps_per_epoch:
            self.current_step += 1
            
            if self.dual_sampling:
                yz_points = torch.rand(self.interior_points, self.dims, device=self.device)
                u_points = torch.rand(self.interior_points, self.dims, device=self.device)
                return yz_points, u_points
            
            single_points = torch.rand(self.interior_points, self.dims, device=self.device)
            return (single_points,)
        else:

            self.current_step = 0
            raise StopIteration
    
    def __len__(self):
        """Return the number of batches per epoch."""
        return self.steps_per_epoch

# Trainer

In [ ]:
class Trainer:
    def __init__(
        self,
        v: nn.Module,
        y: nn.Module,
        theta: nn.Module,
        source: Source,
        train_loader: DataLoader,
        optimizer_v: torch.optim.Optimizer,
        optimizer_y: torch.optim.Optimizer,
        optimizer_theta: torch.optim.Optimizer,
        scheduler_v: torch.optim.lr_scheduler.LambdaLR, 
        scheduler_y: torch.optim.lr_scheduler.LambdaLR,
        scheduler_theta: torch.optim.lr_scheduler.LambdaLR,
        gamma: float = 10.0,
        p: float = 0.5,
        device: torch.device = torch.device("cpu"),
        num_epochs: int = 10000,
        weight: list = [1, 1],
        save: bool = False,
        lr: float = 1e-3,
        eta: float = 0.1,
    ):    
        """
        Args:
            v (nn.Module): Control v(x).
            y (nn.Module): State y(x).
            xi (nn.Module):  Multiplier xi(x).
            source (Source): Source instance.
            train_loader (DataLoader): DataLoader.
            optimizer (torch.optim.Optimizer): Optimizer.
            scheduler (torch.optim.lr_scheduler.LambdaLR): Learning-rate scheduler.
            gamma (float): weight of Moreau envelope.
            p (float): parameter of penalty.
            0 <= p < 1.
            device (torch.device): Training device.
            num_epochs (int): Maximum number of training epochs.
            weight (list): weight of loss.
            save (bool): Whether to save the models.
        """
    
        self.v = v.to(device)
        self.y = y.to(device)
        self.theta = theta.to(device)
        self.source = source
        self.train_loader = train_loader
        self.optimizer_v = optimizer_v
        self.optimizer_y = optimizer_y
        self.optimizer_theta = optimizer_theta
        self.scheduler_v = scheduler_v
        self.scheduler_y = scheduler_y
        self.scheduler_theta = scheduler_theta
        self.gamma = gamma
        self.p = p
        self.device = device
        self.num_epochs = num_epochs
        self.weight = weight
        self.save = save
        self.num = 0
        self.timestart = 0
        self.record_u = torch.zeros(1).to(self.device)
        self.record_l = torch.zeros(1).to(self.device)
        self.record_ortho = torch.zeros(1).to(self.device)
        self.record_non_neg = torch.zeros(1).to(self.device)
        self.lr = lr
        self.eta = eta
        self.file_suffix = f"{datetime.now().strftime('%Y%m%d')}udecay_fixuboundary_gamma_{self.gamma}_cycle_factor_0.8_batch_size{batch_size}_{self.num_epochs}_p_{self.p}_lr_{self.lr}_fixinterior_norestart_eta_{self.eta}_weight_{self.weight[0]}_plusx1"
        if save == True:
            os.makedirs(f'{ROOT}/{self.file_suffix}', exist_ok=True)
        if save == True:
            self.loss_csv_file = f'{ROOT}/{self.file_suffix}/loss.csv'

            with open(self.loss_csv_file, 'w') as f:
                f.write("epoch,time,J_u_loss,J_l_loss,ortho_loss,non_neg_loss\n")


    def diff_net(self, net_y, net_theta):
        """
        Compute the difference between two networks, considering only linear-layer weights and biases.
        """
        res = 0
        for para1, para2 in zip(net_y.parameters(), net_theta.parameters()):
            res += torch.nn.MSELoss(reduction='sum')(para1, para2)
        return res
    
    def getloss_u(self,source,net_y, net_v, x):
        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        return self.weight[0]*J_u(source, net_y, net_v, x)
    
    def getloss_l(self, source, net_y, net_v, x):

        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        

        result = self.weight[1] * J_l(source, net_y, net_v, x, eta=self.eta)
        
        
        return result
    
    def train_one_epoch(self, epoch, timestart):
        self.v.train()
        self.y.train()
        self.theta.train()

        ck = 5*(epoch)**self.p
        inv_ck = 1.0 / ck if ck > 0 else 0.0
        for batch in self.train_loader:
            if isinstance(batch, (tuple, list)):
                if len(batch) == 2:
                    x_yz_raw, x_u_raw = batch
                else:
                    x_yz_raw = batch[0]
                    x_u_raw = batch[0]
            else:
                x_yz_raw = batch
                x_u_raw = batch

            x_yz = x_yz_raw.detach().requires_grad_(True).to(self.device)

            with torch.set_grad_enabled(True):
                theta_params = list(self.theta.parameters())
                y_params = list(self.y.parameters())
                v_params = list(self.v.parameters())
                
                # 1. update theta
                loss_theta_e = self.getloss_l(self.source, self.theta, self.v, x_yz)
                grad_theta_e = torch.autograd.grad(loss_theta_e, theta_params, retain_graph=False)
                theta_total_grads = []
                for g_e, p_theta, p_y in zip(grad_theta_e, theta_params, y_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    theta_total_grads.append(g_e + prox_term)
                self._apply_custom_grads(self.theta, self.optimizer_theta, theta_total_grads)



                # 2. update y and u
                loss_upper_y = self.getloss_u(self.source, self.y, self.v, x_yz)
                loss_lower_y = self.getloss_l(self.source, self.y, self.v, x_yz)
                
                h_j_y = torch.autograd.grad(loss_upper_y, y_params, retain_graph=True)
                h_e_y = torch.autograd.grad(loss_lower_y, y_params, retain_graph=False)
                
                y_total_grads = []
                for g_j, g_e, p_y, p_theta in zip(h_j_y, h_e_y, y_params, theta_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    y_total_grads.append(inv_ck * g_j + g_e + prox_term)

                
                self._apply_custom_grads(self.y, self.optimizer_y, y_total_grads)
                
                x_u = x_u_raw.detach().requires_grad_(True).to(self.device)
                loss_upper_u = self.getloss_u(self.source, self.y, self.v, x_u)
                loss_lower_u = self.getloss_l(self.source, self.y, self.v, x_u)
                loss_lower_theta = self.getloss_l(self.source, self.theta, self.v, x_u)
                
                h_j_u = torch.autograd.grad(loss_upper_u, v_params, retain_graph=True)
                h_e_u = torch.autograd.grad(loss_lower_u, v_params, retain_graph=True)
                h_e_u_theta = torch.autograd.grad(loss_lower_theta, v_params, retain_graph=False)
                
                v_total_grads = []
                for g_j_u, g_e_u, g_e_u_theta in zip(h_j_u, h_e_u, h_e_u_theta):
                    v_total_grads.append(inv_ck * g_j_u + g_e_u - g_e_u_theta)

                
                self._apply_custom_grads(self.v, self.optimizer_v, v_total_grads)



        self.scheduler_v.step()
        self.scheduler_y.step()
        self.scheduler_theta.step()

        

        if (epoch-1) % 10 ==0:        
            # Evaluate the losses on a uniform grid.
            grid_size = 101
            x1 = torch.linspace(0, 1, grid_size, device=self.device)
            x2 = torch.linspace(0, 1, grid_size, device=self.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            self.record_u = J_u(self.source, self.y, self.v, grid_points)
            self.record_l = J_l(self.source, self.y, self.v, grid_points, eta=self.eta)
            self.record_ortho = ortho_loss(self.source, self.y, self.v, grid_points)
            self.record_non_neg = non_neg_loss(self.source, self.y, self.v, grid_points)
            with open(self.loss_csv_file, 'a') as f:
                f.write(f"{epoch},{time.time()-timestart},{self.record_u.item()},{self.record_l.item()},{self.record_ortho.item()},{self.record_non_neg.item()}\n")

            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()


        return self.record_u.item(), self.record_l.item(), self.record_ortho.item(), self.record_non_neg.item()
    

    def _apply_custom_grads(self, model, optimizer, grads):
        """Update model parameters using custom gradients."""
        optimizer.zero_grad()
        for param, grad in zip(model.parameters(), grads):
            param.grad = grad
        optimizer.step()
    
    def validate(self,epoch):
        self.v.eval()
        self.y.eval()
        plt.close('all')
        # REMOVE relative L2 computation and CSV writing
        with torch.no_grad():
            if self.save == True:
                yfile = f'{ROOT}/{self.file_suffix}/y_net_{epoch}.png'
                vfile = f'{ROOT}/{self.file_suffix}/v_net_{epoch}.png'
                xifile = f'{ROOT}/{self.file_suffix}/xi_net_{epoch}.png'
            else:
                yfile = None
                vfile = None
                xifile = None
            visual_y_3d(self.source, self.y, file_name=yfile)
            visual_v_3d(self.source, self.v, file_name=vfile)
        

    def fit(self):
        pbar = tqdm(range(1, self.num_epochs + 2), ascii=True, dynamic_ncols=True)
        self.timestart = time.time()
        clear_output(wait=True)
        for epoch in pbar:

            J_u_loss, J_l_loss, ortho_loss, non_neg_loss = self.train_one_epoch(epoch, self.timestart)


            pbar.set_postfix(
                J_u_loss=J_u_loss,
                J_l_loss=J_l_loss,
                ortho_loss=ortho_loss,
                non_neg_loss=non_neg_loss
            )
            
            if (epoch-1) % 500 == 0:
                self.validate(epoch)
            

# Device

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
source = Source7(device=device)
set_seed(1234)

# Hyperparameters

In [ ]:
# hyperparameters
lr = 2e-4
warmup = 0
cycle_factor=0.8
cycle_length=1000
num_epochs=20000
weight = [1, 10]
weight = [weight[i]/sum(weight) for i in range(len(weight))]
save= True
batch_size = 512
eta = 0.01


In [ ]:
def hard(x1, x2):
    return x1 * (1 - x1) * x2 * (1 - x2)

## Bilevel MEHA training

In [ ]:
dataloader = RandomDataLoader(
    batch_size=batch_size,
    steps_per_epoch=4,
    device=device,
    dual_sampling=True
)
y = net_y_res(source=source,hidden=16,layers=4,activation = 'Swish').to(device)
v = net_v_res(source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard).to(device)
theta = net_y_res(source=source,hidden=16,layers=4,activation = 'Swish').to(device)
theta.load_state_dict(y.state_dict())


optimizer_y = torch.optim.Adam(y.parameters(), lr=lr)
optimizer_v = torch.optim.Adam(v.parameters(), lr=lr)
optimizer_theta = torch.optim.Adam(theta.parameters(), lr=lr)
scheduler_y = LambdaLR(optimizer_y,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
scheduler_v = LambdaLR(optimizer_v,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor**2,cycle_length=cycle_length))
scheduler_theta = LambdaLR(optimizer_theta,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))

trainer = Trainer(
    v=v,
    y=y,
    theta=theta,
    source=source,
    train_loader=dataloader,
    optimizer_v=optimizer_v,
    optimizer_y=optimizer_y,
    optimizer_theta=optimizer_theta,
    scheduler_v=scheduler_v,
    scheduler_y=scheduler_y,
    scheduler_theta=scheduler_theta,
    gamma=200000.0,
    p=0.3,
    device=device,
    num_epochs=num_epochs,
    weight=weight,
    save=save,
    lr = lr,
    eta = eta,
)
trainer.fit()

In [ ]:
import os

# Save the models at the end of bilevel training.
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
os.makedirs(model_dir, exist_ok=True)
torch.save(trainer.y.state_dict(), os.path.join(model_dir, "final_y.pth"))
torch.save(trainer.v.state_dict(), os.path.join(model_dir, "final_v.pth"))
torch.save(trainer.theta.state_dict(), os.path.join(model_dir, "final_theta.pth"))

# Final plots

In [ ]:
def plot_solutions(net_y, net_v, source, device='cuda', save_path=None):
    n_points = 200
    x = torch.linspace(0, 1, n_points, device=device)
    X, Y = torch.meshgrid(x, x, indexing='ij')
    xy = torch.cat((X.reshape(-1, 1), Y.reshape(-1, 1)), dim=1)

    with torch.no_grad():
        y_pred = net_y(xy).reshape(n_points, n_points).cpu().numpy()
        v_pred = net_v(xy).reshape(n_points, n_points).cpu().numpy()

    X_np = X.cpu().numpy()
    Y_np = Y.cpu().numpy()
    save_dir = None
    if save_path:
        save_dir = save_path if os.path.isdir(save_path) else os.path.dirname(save_path)
        save_dir = save_dir if save_dir else save_path
        os.makedirs(save_dir, exist_ok=True)

    def tick_14(ax):
        ax.tick_params(axis='both', which='major', labelsize=14)
    def add_cbar(mappable):
        cbar = plt.colorbar(mappable, pad=0.08, fraction=0.046, shrink=1.0)
        cbar.ax.tick_params(labelsize=14)

    # y plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X_np, Y_np, y_pred, cmap='coolwarm', linewidth=0, antialiased=True)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    tick_14(ax)
    add_cbar(surf)
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'example5_y.pdf'), bbox_inches='tight')
    plt.show()
    plt.close('all')

    # v plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X_np, Y_np, v_pred, cmap='coolwarm', linewidth=0, antialiased=True)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    tick_14(ax)
    add_cbar(surf)
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'example5_u.pdf'), bbox_inches='tight')
    plt.show()
    plt.close('all')

# Load the specified model for Example 6 and plot using legacy example5_y/u.pdf filenames.
# Load the final checkpoints saved above.
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
print(f'Loading models from: {model_dir}')

net_v_loaded = net_v_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard_constraint_2d).to(device)
net_y_loaded = net_y_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard_constraint_2d).to(device)
net_theta_loaded = net_y_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard_constraint_2d).to(device)

net_y_loaded.load_state_dict(torch.load(os.path.join(model_dir, 'final_y.pth'), map_location=device))
net_v_loaded.load_state_dict(torch.load(os.path.join(model_dir, 'final_v.pth'), map_location=device))
net_theta_loaded.load_state_dict(torch.load(os.path.join(model_dir, 'final_theta.pth'), map_location=device))

net_y_loaded.eval()
net_v_loaded.eval()
net_theta_loaded.eval()

solution_dir = os.path.join(ROOT, 'solution_comparison_example5')
os.makedirs(solution_dir, exist_ok=True)

plot_solutions(net_y_loaded, net_v_loaded, source, device=device, save_path=solution_dir)
print(f'Plots saved to: {solution_dir}')